# Chapter 10: Signal Processing

Source span: *Fundamentals of Computer Graphics*, Chapter 10, printed pages 205-254, physical PDF pages 222-271.

## Chapter Goal

Signal processing is the machinery that lets graphics systems move between continuous functions and finite arrays without inventing false structure. This notebook turns the chapter into executable objects: sampled 1D signals, convolution filters, reconstructed functions, synthetic images, spectra, resamplers, and checks that catch aliasing or normalization mistakes.

The central question is practical: when a renderer, image editor, camera model, texture system, or display pipeline stores only samples, which frequencies and averages are preserved, which ones are lost, and which ones come back as artifacts?

## Translation guide

| Source idea | Computational object used here | What can go wrong | Check or invariant |
| --- | --- | --- | --- |
| Continuous signal | Callable function sampled on a dense grid for plotting | The plotted curve is mistaken for the stored data | Samples are stored separately from the dense reference curve |
| Sampling | Values on an integer or lattice grid | A high frequency masquerades as a low frequency | Aliased cosine samples match to numerical precision |
| Reconstruction | Discrete-continuous convolution with a filter | Stair steps, ripple, blur, or ringing | Constant signals stay constant after normalized reconstruction |
| Discrete convolution | Weighted sum over neighboring samples | Kernel flipped or normalized incorrectly | Commutativity, associativity, identity, and sum-one checks |
| Filter family | Box, tent, Gaussian, B-spline, Catmull-Rom, Mitchell-Netravali, and windowed sinc functions | Wrong support, negative lobes, or non-unit area | Area, interpolation, ripple, and negative-lobe ledger |
| Image filtering | 2D convolution on a synthetic image | Boundary darkening or accidental brightness change | Blurs reduce variance; separable filtering matches a full 2D kernel |
| Antialiasing | Low-pass filtering before sampling | Moire patterns and jagged high-frequency content | High-frequency energy falls after scaled filtering |
| Frequency response | FFT magnitude and Fourier-domain copies | Base spectrum and alias spectra overlap | Filtered overlap energy is lower than unfiltered overlap energy |
| Resampling | Reconstruction and sampling folded into one scaled filter | Downsampling with a reconstruction-sized filter aliases | Scaled Mitchell resampling has lower high-frequency energy than nearest sampling |

## Visual storyboard

| Order | Artifact | Concept | Representation and library | Learner inspection target | Validation |
| --- | --- | --- | --- | --- | --- |
| 1 | `sampling-aliasing-sine-reconstruction.png` | Sampling and reconstruction artifacts | Matplotlib 1D plots | Same low-rate samples can come from two different cosines; zero-order hold creates reconstruction steps | Low-rate high and aliased samples are equal within tolerance |
| 2 | `convolution-shifted-filter-sum.png` | Convolution as weighted averages and shifted filters | NumPy/SciPy convolution with Matplotlib bars | One output sample is a local weighted average; the whole output is a sum of shifted kernels | Commutativity, associativity, identity, and step-ramp checks |
| 3 | `reconstruction-filter-property-gallery.png` and `filter-property-ledger.csv` | Filter shapes and reconstruction properties | Analytic filter functions plus pandas ledger | Support, smoothness, interpolation, negative lobes, and ripple behavior | Numeric area, ripple span, and interpolation errors |
| 4 | `frequency-response-aliasing-nyquist.png` and `sampling-spectrum-alias-lab.html` | Fourier response, Nyquist limit, and alias spectra | FFT/analytic spectra with Matplotlib and Plotly HTML | Smooth filters shrink high-frequency content; sample-rate copies overlap when beyond Nyquist | Filtered alias-overlap energy is lower than unfiltered alias-overlap energy |
| 5 | `synthetic-image-filtering-operations.png` | Image processing by convolution | Synthetic image plus SciPy ndimage | Blur, unsharp masking, edge/detail extraction, and soft shadow as shifted blur | Blur variance decreases; unsharp high-pass energy increases |
| 6 | `resampling-antialiasing-comparison.png` | Filtering and resampling images | Chapter-local separable resampler, FFT spectra | Pixel dropping keeps false high frequencies; scaled filters suppress moire before decimation | Scaled Mitchell high-frequency energy is lower than nearest sampling |

Library routing: Matplotlib is used for durable labeled figures; SciPy is used for convolution, FFT, and image filtering; scikit-image is available but the core resampler is written directly so the reconstruction/sampling model stays visible; Plotly is used where toggling spectral traces makes the frequency-domain argument easier to inspect; pandas records the filter property ledger.

In [ ]:
from pathlib import Path
import sys

CHAPTER = 10
TOPIC = f"chapter-{CHAPTER:02d}"
TITLE = "Signal Processing"
PRINTED_PAGES = "205-254"
PDF_PAGES = "222-271"

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the FCG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
FIG_DIR = ARTIFACT_ROOT / "figures"
HTML_DIR = ARTIFACT_ROOT / "html"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
DATA_DIR = ARTIFACT_ROOT / "data"
for directory in [FIG_DIR, HTML_DIR, CHECK_DIR, TABLE_DIR, DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import ndimage, signal as sp_signal
from scipy.fft import fft2, fftfreq, fftshift, irfft, rfft

from utils.artifacts import assert_artifacts, display_artifact, save_json, save_matplotlib, save_plotly_html, save_table_csv
from utils.notebook_checks import assert_nonblank_image
from utils.plotting import PALETTE, close, style_axis

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
})

figure_paths = []
html_paths = []
check_paths = []
table_paths = []
invariant_checks = {"source_span": {"printed_pages": PRINTED_PAGES, "pdf_pages": PDF_PAGES}}

visual_storyboard = [
    {"artifact": "sampling-aliasing-sine-reconstruction.png", "concept": "sampling and reconstruction", "library": "matplotlib", "validation": "aliased samples match"},
    {"artifact": "convolution-shifted-filter-sum.png", "concept": "weighted average and shifted-filter views", "library": "numpy/scipy/matplotlib", "validation": "convolution algebra checks"},
    {"artifact": "reconstruction-filter-property-gallery.png", "concept": "filter impulse responses and reconstruction properties", "library": "numpy/matplotlib/pandas", "validation": "area, ripple, interpolation ledger"},
    {"artifact": "frequency-response-aliasing-nyquist.png", "concept": "frequency response and Nyquist alias overlap", "library": "numpy FFT/matplotlib", "validation": "filtered overlap is lower"},
    {"artifact": "synthetic-image-filtering-operations.png", "concept": "image convolution operations", "library": "scipy.ndimage/matplotlib", "validation": "variance and high-pass checks"},
    {"artifact": "resampling-antialiasing-comparison.png", "concept": "resampling filters and alias suppression", "library": "chapter-local separable resampler/matplotlib", "validation": "scaled filter high-frequency energy"},
    {"artifact": "sampling-spectrum-alias-lab.html", "concept": "interactive Fourier-domain sampling view", "library": "plotly", "validation": "HTML artifact exists"},
]
storyboard_path = save_json(visual_storyboard, TOPIC, "visual-storyboard.json")
check_paths.append(storyboard_path)

def normalize01(values):
    arr = np.asarray(values, dtype=float)
    lo = float(np.nanmin(arr))
    hi = float(np.nanmax(arr))
    if hi - lo < 1e-12:
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)

def book_path(path):
    p = Path(path)
    if not p.is_absolute() and p.parts and p.parts[0] == "artifacts":
        return p.as_posix()
    try:
        return p.resolve().relative_to(BOOK_ROOT).as_posix()
    except ValueError:
        return p.as_posix()

def box_filter(x, radius=0.5):
    x = np.asarray(x, dtype=float)
    return np.where(np.abs(x) < radius, 1.0 / (2.0 * radius), 0.0)

def tent_filter(x):
    x = np.asarray(x, dtype=float)
    return np.maximum(1.0 - np.abs(x), 0.0)

def gaussian_filter_1d(x, sigma=1.0):
    x = np.asarray(x, dtype=float)
    return np.exp(-0.5 * (x / sigma) ** 2) / (sigma * np.sqrt(2.0 * np.pi))

def b_spline_filter(x):
    x = np.asarray(x, dtype=float)
    a = np.abs(x)
    out = np.zeros_like(a)
    mask0 = a < 1.0
    mask1 = (a >= 1.0) & (a < 2.0)
    out[mask0] = (4.0 - 6.0 * a[mask0] ** 2 + 3.0 * a[mask0] ** 3) / 6.0
    out[mask1] = (2.0 - a[mask1]) ** 3 / 6.0
    return out

def catmull_rom_filter(x):
    x = np.asarray(x, dtype=float)
    a = np.abs(x)
    out = np.zeros_like(a)
    mask0 = a < 1.0
    mask1 = (a >= 1.0) & (a < 2.0)
    out[mask0] = 1.5 * a[mask0] ** 3 - 2.5 * a[mask0] ** 2 + 1.0
    out[mask1] = -0.5 * a[mask1] ** 3 + 2.5 * a[mask1] ** 2 - 4.0 * a[mask1] + 2.0
    return out

def mitchell_netravali_filter(x):
    return (b_spline_filter(x) + 2.0 * catmull_rom_filter(x)) / 3.0

def lanczos3_filter(x):
    x = np.asarray(x, dtype=float)
    a = 3.0
    ax = np.abs(x)
    return np.where(ax < a, np.sinc(x) * np.sinc(x / a), 0.0)

FILTERS = {
    "box": {"fn": lambda x: box_filter(x, 0.5), "radius": 0.5, "finite": True, "continuity": "discontinuous"},
    "tent": {"fn": tent_filter, "radius": 1.0, "finite": True, "continuity": "C0"},
    "Gaussian sigma=0.5": {"fn": lambda x: gaussian_filter_1d(x, 0.5), "radius": 2.0, "finite": False, "continuity": "smooth"},
    "B-spline cubic": {"fn": b_spline_filter, "radius": 2.0, "finite": True, "continuity": "C2"},
    "Catmull-Rom cubic": {"fn": catmull_rom_filter, "radius": 2.0, "finite": True, "continuity": "C1"},
    "Mitchell-Netravali": {"fn": mitchell_netravali_filter, "radius": 2.0, "finite": True, "continuity": "C1"},
    "Lanczos-3 sinc": {"fn": lanczos3_filter, "radius": 3.0, "finite": True, "continuity": "windowed sinc"},
}

def scaled_filter_values(fn, x, scale):
    scale = float(scale)
    return fn(np.asarray(x, dtype=float) / scale) / scale

def reconstruction_weights(x, radius, fn, scale=1.0, n=None):
    support = radius * scale
    lo = int(math.ceil(x - support))
    hi = int(math.floor(x + support))
    idx = np.arange(lo, hi + 1)
    if idx.size == 0:
        idx = np.array([int(round(x))])
    weights = scaled_filter_values(fn, x - idx, scale)
    if n is not None:
        keep = (idx >= 0) & (idx < n)
        idx = idx[keep]
        weights = weights[keep]
    s = float(weights.sum())
    if abs(s) > 1e-12:
        weights = weights / s
    return idx, weights

def reconstruct_1d(samples, xs, fn, radius, scale=1.0, normalize=True):
    samples = np.asarray(samples, dtype=float)
    out = []
    for x in np.asarray(xs, dtype=float):
        idx, weights = reconstruction_weights(x, radius, fn, scale=scale, n=len(samples))
        if not normalize:
            weights = scaled_filter_values(fn, x - idx, scale)
        out.append(float(np.dot(samples[idx], weights)))
    return np.array(out)

def resample_axis(arr, new_length, axis, fn, radius, filter_scale=None):
    arr = np.asarray(arr, dtype=float)
    old_length = arr.shape[axis]
    xl, xh = -0.5, old_length - 0.5
    delta = (xh - xl) / new_length
    scale = max(1.0, delta) if filter_scale is None else float(filter_scale)
    support = radius * scale
    coords = xl + delta * (np.arange(new_length) + 0.5)
    moved = np.moveaxis(arr, axis, 0)
    out = np.empty((new_length,) + moved.shape[1:], dtype=float)
    for oi, x in enumerate(coords):
        lo = int(math.ceil(x - support))
        hi = int(math.floor(x + support))
        idx = np.arange(lo, hi + 1)
        if idx.size == 0:
            idx = np.array([int(round(x))])
        weights = scaled_filter_values(fn, x - idx, scale)
        idx_clipped = np.clip(idx, 0, old_length - 1)
        s = float(weights.sum())
        if abs(s) > 1e-12:
            weights = weights / s
        out[oi] = np.tensordot(weights, moved[idx_clipped], axes=(0, 0))
    return np.moveaxis(out, 0, axis)

def resample_image(image, out_shape, fn, radius, filter_scale=None):
    tmp = resample_axis(image, out_shape[1], axis=1, fn=fn, radius=radius, filter_scale=filter_scale)
    out = resample_axis(tmp, out_shape[0], axis=0, fn=fn, radius=radius, filter_scale=filter_scale)
    return np.clip(out, 0.0, 1.0)

def fft_response(fn, radius=6.0, n=4096):
    x = np.linspace(-radius, radius, n, endpoint=False)
    dx = float(x[1] - x[0])
    values = fn(x)
    spec = np.fft.fftshift(np.fft.fft(np.fft.ifftshift(values))) * dx
    freq = np.fft.fftshift(np.fft.fftfreq(n, d=dx))
    mag = np.abs(spec)
    if mag.max() > 0:
        mag = mag / mag.max()
    return freq, mag

def high_frequency_energy(image, threshold=0.34):
    image = np.asarray(image, dtype=float)
    h, w = image.shape[:2]
    centered = image - image.mean(axis=(0, 1), keepdims=True)
    if centered.ndim == 3:
        centered = centered.mean(axis=2)
    fy = fftshift(fftfreq(h))
    fx = fftshift(fftfreq(w))
    FX, FY = np.meshgrid(fx, fy)
    radius = np.sqrt(FX**2 + FY**2)
    power = np.abs(fftshift(fft2(centered))) ** 2
    total = float(power.sum())
    return float(power[radius > threshold].sum() / total) if total > 0 else 0.0

## 1. Sampling: one set of numbers can fit two stories

A sampled signal is not the original continuous function. It is a sequence of measurements. If the sample rate is too low, a high-frequency input can produce exactly the same sample values as a lower-frequency signal. After that point, no reconstruction algorithm can know which story was true.

The figure below keeps the dense continuous cosine separate from the stored samples. The upper panel shows a comfortable sample rate. The middle panel uses a sample rate of 10 for a frequency 7 cosine; the same samples also fit a frequency 3 cosine. The lower panel shows two reconstruction choices from the same numbers: zero-order hold, which resembles a DAC staircase, and tent/linear reconstruction, which removes the hard steps but cannot recover the lost high frequency.

In [ ]:
t = np.linspace(0.0, 1.0, 2400, endpoint=False)
true_freq = 7.0
fs_good = 32.0
fs_low = 10.0
alias_freq = abs(true_freq - round(true_freq / fs_low) * fs_low)

good_n = np.arange(0, int(fs_good), dtype=float)
good_t = good_n / fs_good
low_n = np.arange(0, int(fs_low), dtype=float)
low_t = low_n / fs_low
continuous = np.cos(2.0 * np.pi * true_freq * t)
alias_curve = np.cos(2.0 * np.pi * alias_freq * t)
good_samples = np.cos(2.0 * np.pi * true_freq * good_t)
low_samples = np.cos(2.0 * np.pi * true_freq * low_t)
alias_samples = np.cos(2.0 * np.pi * alias_freq * low_t)

x_dense = np.linspace(0, len(low_samples) - 1, 900)
zero_hold = low_samples[np.clip(np.floor(x_dense + 1e-12).astype(int), 0, len(low_samples) - 1)]
tent_recon = np.interp(x_dense, np.arange(len(low_samples)), low_samples)

fig, axes = plt.subplots(3, 1, figsize=(10.5, 8.6), constrained_layout=True)
ax = axes[0]
ax.plot(t, continuous, color=PALETTE["blue"], lw=1.7, label="continuous 7 cycles/sec")
ax.scatter(good_t, good_samples, s=28, color=PALETTE["ink"], zorder=3, label="32 samples/sec")
style_axis(ax, "High enough sample rate: samples trace the intended cosine", xlabel="time", ylabel="value")
ax.legend(loc="upper right")

ax = axes[1]
ax.plot(t, continuous, color=PALETTE["blue"], lw=1.5, label="7 cycles/sec original")
ax.plot(t, alias_curve, color=PALETTE["red"], lw=1.4, ls="--", label="3 cycles/sec alias")
ax.scatter(low_t, low_samples, s=36, color=PALETTE["ink"], zorder=4, label="10 samples/sec")
ax.axhline(0, color="#98a2ad", lw=0.8)
style_axis(ax, "Undersampling: the two curves agree at every stored sample", xlabel="time", ylabel="value")
ax.legend(loc="upper right")

ax = axes[2]
ax.step(x_dense, zero_hold, where="post", color=PALETTE["gold"], lw=1.6, label="box / zero-order hold")
ax.plot(x_dense, tent_recon, color=PALETTE["teal"], lw=1.8, label="tent / linear reconstruction")
ax.scatter(np.arange(len(low_samples)), low_samples, s=32, color=PALETTE["ink"], zorder=4, label="stored samples")
style_axis(ax, "Reconstruction changes the between-sample story, not the samples", xlabel="sample index coordinate", ylabel="value")
ax.legend(loc="upper right")

sampling_path = save_matplotlib(fig, TOPIC, "sampling-aliasing-sine-reconstruction.png")
close(fig)
figure_paths.append(sampling_path)

sampling_checks = {
    "true_frequency": true_freq,
    "low_sample_rate": fs_low,
    "alias_frequency": alias_freq,
    "alias_sample_max_error": float(np.max(np.abs(low_samples - alias_samples))),
    "zero_order_hold_jump_count": int(np.count_nonzero(np.diff(zero_hold) != 0.0)),
    "tent_reconstruction_sample_error": float(np.max(np.abs(np.interp(np.arange(len(low_samples)), x_dense, tent_recon) - low_samples))),
}
invariant_checks["sampling"] = sampling_checks
assert sampling_checks["alias_sample_max_error"] < 1e-12
display_artifact(sampling_path, width=860)
sampling_checks

## 2. Convolution: local averages and shifted copies are the same operation

The chapter uses convolution in several forms: discrete-discrete filtering, continuous filtering, reconstruction of a discrete sequence by a continuous filter, and 2D image filtering. The discrete form is already enough to see the two essential interpretations.

First, a single output sample is a weighted average of nearby input samples. Second, the whole output is a sum of shifted copies of the filter, one copy per nonzero input sample. These two views are algebraically identical, and the checks below use that algebra directly: convolution is commutative, associative, distributive over addition, and has an impulse identity.

In [ ]:
indices = np.arange(-10, 15)
base_signal = np.where(indices >= 0, 1.0, 0.0) + 0.2 * np.exp(-0.5 * ((indices - 7) / 1.4) ** 2)
kernel_offsets = np.arange(-2, 3)
binomial_kernel = np.array([1, 4, 6, 4, 1], dtype=float) / 16.0
convolved = sp_signal.convolve(base_signal, binomial_kernel, mode="same")
focus_i = 2
focus_pos = np.where(indices == focus_i)[0][0]
local_positions = focus_pos + kernel_offsets
local_values = base_signal[local_positions]
local_weights = binomial_kernel[::-1]
weighted_products = local_values * local_weights

impulse_positions = np.array([-5, 0, 4, 8])
impulse_weights = np.array([0.55, 1.0, 0.65, 0.35])
x_shift = np.arange(-10, 15)
shifted_sum = np.zeros_like(x_shift, dtype=float)
shifted_components = []
for pos, weight in zip(impulse_positions, impulse_weights):
    component = weight * np.interp(x_shift - pos, kernel_offsets, binomial_kernel, left=0.0, right=0.0)
    shifted_components.append(component)
    shifted_sum += component

fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.4), constrained_layout=True)
ax = axes[0, 0]
ax.bar(indices, base_signal, color="#d7e6f7", edgecolor=PALETTE["blue"], label="input samples")
ax.bar(indices[local_positions], local_values, color=PALETTE["blue"], alpha=0.75, label="samples used at i=2")
ax2 = ax.twinx()
ax2.plot(indices[local_positions], local_weights, "o-", color=PALETTE["red"], label="flipped kernel weights")
ax2.set_ylabel("weight")
ax2.set_ylim(0, 0.48)
style_axis(ax, "One output sample is a weighted local average", xlabel="sample index", ylabel="sample value")
ax.legend(loc="upper left")
ax2.legend(loc="upper right")

ax = axes[0, 1]
ax.bar(indices[local_positions], weighted_products, color=PALETTE["teal"], edgecolor="white")
ax.axhline(weighted_products.sum(), color=PALETTE["red"], lw=1.5, label=f"sum = {weighted_products.sum():.3f}")
style_axis(ax, "Products that sum to (a * b)[2]", xlabel="sample index", ylabel="value times weight")
ax.legend(loc="upper left")

ax = axes[1, 0]
ax.plot(indices, base_signal, "o-", color=PALETTE["gray"], label="input")
ax.plot(indices, convolved, "o-", color=PALETTE["blue"], label="input convolved with binomial kernel")
style_axis(ax, "A normalized filter smooths a step without changing flat regions", xlabel="sample index", ylabel="value")
ax.legend(loc="lower right")

ax = axes[1, 1]
for component, pos, weight in zip(shifted_components, impulse_positions, impulse_weights):
    ax.plot(x_shift, component, color="#91b8dd", alpha=0.8, lw=1.2)
    ax.text(pos, component.max() + 0.015, f"{weight:.2g}", ha="center", fontsize=8, color=PALETTE["ink"])
ax.plot(x_shift, shifted_sum, color=PALETTE["red"], lw=2.0, label="sum of shifted filters")
ax.scatter(impulse_positions, impulse_weights * 0.38, color=PALETTE["ink"], s=28, label="input impulses")
style_axis(ax, "Whole convolution as shifted copies of the filter", xlabel="sample index", ylabel="value")
ax.legend(loc="upper right")

convolution_path = save_matplotlib(fig, TOPIC, "convolution-shifted-filter-sum.png")
close(fig)
figure_paths.append(convolution_path)

a = np.array([0.0, 2.0, -1.0, 4.0, 1.5])
b = np.array([0.25, 0.5, 0.25])
c = np.array([-1.0, 0.0, 1.0])
conv_ab = sp_signal.convolve(a, b, mode="full")
conv_ba = sp_signal.convolve(b, a, mode="full")
assoc_left = sp_signal.convolve(sp_signal.convolve(a, b, mode="full"), c, mode="full")
assoc_right = sp_signal.convolve(a, sp_signal.convolve(b, c, mode="full"), mode="full")
impulse = np.array([1.0])
step_idx = np.arange(-7, 8)
step = (step_idx >= 0).astype(float)
box5 = np.ones(5) / 5.0
step_box = sp_signal.convolve(step, box5, mode="same")
ramp_window = step_box[(step_idx >= -2) & (step_idx <= 2)]

convolution_checks = {
    "kernel_sum": float(binomial_kernel.sum()),
    "local_weighted_average": float(weighted_products.sum()),
    "same_value_from_convolution_array": float(convolved[focus_pos]),
    "commutative_max_error": float(np.max(np.abs(conv_ab - conv_ba))),
    "associative_max_error": float(np.max(np.abs(assoc_left - assoc_right))),
    "identity_max_error": float(np.max(np.abs(sp_signal.convolve(a, impulse, mode="full") - a))),
    "box_step_ramp_values": [float(x) for x in ramp_window],
}
invariant_checks["convolution"] = convolution_checks
assert convolution_checks["commutative_max_error"] < 1e-12
assert convolution_checks["associative_max_error"] < 1e-12
assert convolution_checks["identity_max_error"] < 1e-12
assert abs(convolution_checks["kernel_sum"] - 1.0) < 1e-12
display_artifact(convolution_path, width=900)
convolution_checks

## 3. Reconstruction filters: support, smoothness, interpolation, ripple, and ringing

A filter is more than a curve. Its support controls cost, its area controls average brightness, its smoothness controls reconstructed continuity, its integer samples determine whether it interpolates, and its negative lobes predict ringing near discontinuities.

The source span emphasizes that no single useful filter is perfect. Box and tent filters are cheap. Gaussian filters are strong sampling filters because their frequency response decays smoothly. B-spline filters are smooth but not interpolating. Catmull-Rom interpolates but rings because it has negative lobes. Mitchell-Netravali sits between the two cubic filters and is a practical image-resampling compromise. Lanczos is a windowed version of the ideal sinc idea: good frequency selectivity, but visible ringing is possible.

In [ ]:
x = np.linspace(-4.0, 4.0, 2400)
filter_colors = {
    "box": PALETTE["gray"],
    "tent": PALETTE["blue"],
    "Gaussian sigma=0.5": PALETTE["green"],
    "B-spline cubic": PALETTE["teal"],
    "Catmull-Rom cubic": PALETTE["red"],
    "Mitchell-Netravali": PALETTE["violet"],
    "Lanczos-3 sinc": PALETTE["gold"],
}

rows = []
for name, spec in FILTERS.items():
    fn = spec["fn"]
    radius = spec["radius"]
    values = fn(x)
    area = float(np.trapezoid(values, x))
    integer_offsets = np.arange(-5, 6)
    integer_values = fn(integer_offsets)
    target = (integer_offsets == 0).astype(float)
    interpolation_error = float(np.max(np.abs(integer_values - target)))
    xr = np.linspace(0.0, 1.0, 401)
    shifts = np.arange(-12, 13)
    ripple_values = np.array([fn(xx - shifts).sum() for xx in xr])
    rows.append({
        "filter": name,
        "natural_radius": radius,
        "finite_support": bool(spec["finite"]),
        "continuity": spec["continuity"],
        "area_on_plotted_support": area,
        "interpolating_error": interpolation_error,
        "minimum_value": float(values.min()),
        "ripple_sum_min": float(ripple_values.min()),
        "ripple_sum_max": float(ripple_values.max()),
        "ripple_span": float(ripple_values.max() - ripple_values.min()),
    })

property_rows = rows
property_table_path = save_table_csv(property_rows, TOPIC, "filter-property-ledger.csv")
table_paths.append(property_table_path)
property_df = pd.DataFrame(property_rows)

fig, axes = plt.subplots(2, 2, figsize=(12.0, 7.6), constrained_layout=True)
ax = axes[0, 0]
for name, spec in FILTERS.items():
    ax.plot(x, spec["fn"](x), lw=1.6, color=filter_colors[name], label=name)
style_axis(ax, "Spatial impulse responses", xlabel="x in input-sample units", ylabel="filter value")
ax.set_xlim(-3.25, 3.25)
ax.legend(ncols=2, loc="upper right")

ax = axes[0, 1]
for name in ["box", "tent", "B-spline cubic", "Catmull-Rom cubic", "Mitchell-Netravali"]:
    spec = FILTERS[name]
    integer_offsets = np.arange(-3, 4)
    ax.plot(integer_offsets, spec["fn"](integer_offsets), "o-", color=filter_colors[name], label=name)
style_axis(ax, "Integer samples determine interpolation", xlabel="integer offset from sample", ylabel="filter value")
ax.legend(loc="upper right")

ax = axes[1, 0]
for name in ["box", "tent", "Gaussian sigma=0.5", "B-spline cubic", "Mitchell-Netravali"]:
    spec = FILTERS[name]
    freq, mag = fft_response(spec["fn"], radius=8.0)
    keep = np.abs(freq) <= 2.5
    ax.plot(freq[keep], mag[keep], lw=1.5, color=filter_colors[name], label=name)
ax.axvline(0.5, color=PALETTE["red"], lw=1.0, ls="--", label="Nyquist for unit samples")
ax.axvline(-0.5, color=PALETTE["red"], lw=1.0, ls="--")
style_axis(ax, "Magnitude response: smoothing is low-pass filtering", xlabel="frequency, cycles per sample", ylabel="normalized magnitude")
ax.legend(loc="upper right")

ax = axes[1, 1]
xr = np.linspace(0.0, 1.0, 400)
shifts = np.arange(-12, 13)
for name in ["box", "tent", "Gaussian sigma=0.5", "B-spline cubic", "Catmull-Rom cubic", "Mitchell-Netravali"]:
    spec = FILTERS[name]
    ripple_values = np.array([spec["fn"](xx - shifts).sum() for xx in xr])
    ax.plot(xr, ripple_values, lw=1.5, color=filter_colors[name], label=name)
ax.axhline(1.0, color="#8994a3", lw=0.9)
style_axis(ax, "Ripple-free reconstruction keeps constants constant", xlabel="fractional position between samples", ylabel="sum of weights")
ax.legend(ncols=2, loc="lower center")

filter_gallery_path = save_matplotlib(fig, TOPIC, "reconstruction-filter-property-gallery.png")
close(fig)
figure_paths.append(filter_gallery_path)

filter_checks = {
    "max_unit_area_error_finite_or_plotted": float(max(abs(row["area_on_plotted_support"] - 1.0) for row in property_rows if row["filter"] != "Lanczos-3 sinc")),
    "catmull_rom_minimum_value": float(property_df.loc[property_df["filter"] == "Catmull-Rom cubic", "minimum_value"].iloc[0]),
    "b_spline_interpolation_error": float(property_df.loc[property_df["filter"] == "B-spline cubic", "interpolating_error"].iloc[0]),
    "tent_ripple_span": float(property_df.loc[property_df["filter"] == "tent", "ripple_span"].iloc[0]),
    "gaussian_ripple_span": float(property_df.loc[property_df["filter"] == "Gaussian sigma=0.5", "ripple_span"].iloc[0]),
}
invariant_checks["filters"] = filter_checks
assert filter_checks["catmull_rom_minimum_value"] < 0.0
assert filter_checks["b_spline_interpolation_error"] > 0.1
assert filter_checks["tent_ripple_span"] < 1e-12

display_artifact(filter_gallery_path, width=920)
display(property_df.round(5))
display_artifact(property_table_path)
filter_checks

## 4. Frequency response: aliasing is spectrum overlap

The Fourier transform gives convolution a simpler description: filtering in the spatial domain multiplies the spectrum by the filter response. Sampling multiplies by an impulse train in space, which creates repeated copies of the spectrum in frequency. If those copies overlap the base spectrum, distinct frequencies become mixed into the same samples.

The next figure uses a synthetic spectrum with both smooth low-frequency content and a narrow high-frequency feature. Sampling without filtering repeats that spectrum too closely. A low-pass filter narrows the spectrum before sampling. Increasing the sample rate moves the copies farther apart. The notebook records a numeric overlap measure so the plot is not just qualitative.

In [ ]:
u = np.linspace(-3.0, 3.0, 4000)

def source_spectrum(freq):
    freq = np.asarray(freq, dtype=float)
    low = np.exp(-0.5 * (freq / 0.28) ** 2)
    high = 0.32 * np.exp(-0.5 * ((np.abs(freq) - 0.78) / 0.07) ** 2)
    return low + high

def gaussian_lowpass_response(freq, cutoff=0.42):
    return np.exp(-0.5 * (freq / cutoff) ** 2)

def alias_components(sample_rate, filtered=False):
    base = source_spectrum(u)
    if filtered:
        base = base * gaussian_lowpass_response(u, cutoff=0.38)
    copies = []
    for k in range(-4, 5):
        shifted = source_spectrum(u - k * sample_rate)
        if filtered:
            shifted = shifted * gaussian_lowpass_response(u - k * sample_rate, cutoff=0.38)
        copies.append((k, shifted))
    alias = sum(arr for k, arr in copies if k != 0)
    total = sum(arr for _, arr in copies)
    return base, alias, total

fs_unit = 1.0
base_unfiltered, alias_unfiltered, total_unfiltered = alias_components(fs_unit, filtered=False)
base_filtered, alias_filtered, total_filtered = alias_components(fs_unit, filtered=True)
base_fast, alias_fast, total_fast = alias_components(2.0, filtered=False)
passband = np.abs(u) <= fs_unit / 2.0
overlap_unfiltered = float(np.trapezoid(alias_unfiltered[passband], u[passband]))
overlap_filtered = float(np.trapezoid(alias_filtered[passband], u[passband]))
overlap_fast = float(np.trapezoid(alias_fast[passband], u[passband]))

fig, axes = plt.subplots(3, 1, figsize=(10.8, 8.4), constrained_layout=True, sharex=True)
panels = [
    ("Sampling at 1 sample/unit without low-pass filtering", base_unfiltered, alias_unfiltered, total_unfiltered, overlap_unfiltered),
    ("Same sample rate after a Gaussian low-pass filter", base_filtered, alias_filtered, total_filtered, overlap_filtered),
    ("Higher sample rate: spectrum copies move apart", base_fast, alias_fast, total_fast, overlap_fast),
]
for ax, (title, base, alias, total, overlap) in zip(axes, panels):
    ax.fill_between(u, 0, alias, color="#f2b8b5", alpha=0.75, label="alias copies inside view")
    ax.plot(u, base, color=PALETTE["blue"], lw=1.9, label="base spectrum")
    ax.plot(u, total, color=PALETTE["ink"], lw=1.0, alpha=0.7, label="base + aliases")
    ax.axvspan(-0.5, 0.5, color="#e8f1fb", alpha=0.7, label="unit-rate Nyquist band" if ax is axes[0] else None)
    ax.axvline(-0.5, color=PALETTE["red"], ls="--", lw=0.9)
    ax.axvline(0.5, color=PALETTE["red"], ls="--", lw=0.9)
    style_axis(ax, f"{title} (alias overlap {overlap:.3f})", ylabel="magnitude")
    ax.set_ylim(0, 1.65)
axes[-1].set_xlabel("frequency, cycles per unit")
axes[0].legend(ncols=3, loc="upper right")

freq_path = save_matplotlib(fig, TOPIC, "frequency-response-aliasing-nyquist.png")
close(fig)
figure_paths.append(freq_path)

frequency_checks = {
    "unit_sample_rate_alias_overlap": overlap_unfiltered,
    "lowpass_alias_overlap": overlap_filtered,
    "double_rate_alias_overlap": overlap_fast,
    "lowpass_overlap_ratio": float(overlap_filtered / overlap_unfiltered),
    "double_rate_overlap_ratio": float(overlap_fast / overlap_unfiltered),
}
invariant_checks["frequency_response"] = frequency_checks
assert overlap_filtered < 0.55 * overlap_unfiltered
assert overlap_fast < overlap_unfiltered
display_artifact(freq_path, width=880)
frequency_checks

## 5. Image processing: 2D convolution uses the same algebra

Images are sampled 2D signals. A blur is a low-pass filter; sharpening is the original image plus a scaled high-pass detail image; a soft drop shadow is a shifted mask convolved with a blur. The point of using a synthetic image here is control: edges, ramps, rings, and stripes are all known before filtering, so the measured changes are meaningful.

The separable-filter idea matters for performance. A 2D Gaussian formed as an outer product of a 1D kernel can be applied as rows then columns. The checks compare that separable result with a full 2D convolution on a small patch.

In [ ]:
def synthetic_graphics_image(n=256):
    y, x = np.mgrid[0:n, 0:n]
    X = (x - n / 2) / (n / 2)
    Y = (y - n / 2) / (n / 2)
    rings = 0.5 + 0.5 * np.cos(44.0 * (X**2 + Y**2))
    diagonal = 0.5 + 0.5 * np.sin(30.0 * (0.78 * X + 0.28 * Y))
    ramp = 0.25 + 0.45 * (X + 1.0) / 2.0
    edge = (X + 0.35 * Y > -0.08).astype(float)
    disk = ((X + 0.42) ** 2 + (Y - 0.28) ** 2 < 0.12).astype(float)
    image = 0.32 * rings + 0.23 * diagonal + 0.25 * ramp + 0.20 * edge
    image = np.maximum(image, disk * 0.92)
    return np.clip(image, 0.0, 1.0)

image = synthetic_graphics_image(256)
box7 = np.ones((7, 7), dtype=float) / 49.0
box_blur = ndimage.convolve(image, box7, mode="nearest")
gaussian_blur = ndimage.gaussian_filter(image, sigma=2.2, mode="nearest")
highpass = image - gaussian_blur
unsharp = np.clip(image + 1.35 * highpass, 0.0, 1.0)
sobel = normalize01(np.hypot(ndimage.sobel(image, axis=0), ndimage.sobel(image, axis=1)))

yy, xx = np.mgrid[0:256, 0:256]
mask = (((xx - 95) / 54) ** 2 + ((yy - 90) / 38) ** 2 < 1.0) | ((xx > 118) & (xx < 184) & (yy > 128) & (yy < 177))
shifted_mask = ndimage.shift(mask.astype(float), shift=(18, 22), order=0, mode="constant", cval=0.0)
shadow = ndimage.gaussian_filter(shifted_mask, sigma=7.0, mode="constant")
object_layer = mask.astype(float)
shadow_scene = np.clip(0.86 - 0.46 * shadow + 0.18 * image, 0.0, 1.0)
shadow_scene = np.where(object_layer > 0, 0.08 + 0.82 * image, shadow_scene)

fig, axes = plt.subplots(2, 3, figsize=(11.2, 7.5), constrained_layout=True)
panels = [
    (image, "synthetic input", "Known edges, rings, stripes"),
    (box_blur, "box blur", "Cheap average, blockier response"),
    (gaussian_blur, "Gaussian blur", "Smoother low-pass filter"),
    (unsharp, "unsharp mask", "Original + high-pass detail"),
    (sobel, "edge/detail magnitude", "High frequencies made visible"),
    (shadow_scene, "shift + Gaussian blur", "Soft shadow as convolution"),
]
for ax, (arr, title, note) in zip(axes.ravel(), panels):
    ax.imshow(arr, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.text(0.02, 0.98, note, transform=ax.transAxes, ha="left", va="top", fontsize=8, color=PALETTE["ink"], bbox={"facecolor": "white", "edgecolor": "#c9d3df", "alpha": 0.88, "pad": 2})

image_filter_path = save_matplotlib(fig, TOPIC, "synthetic-image-filtering-operations.png")
close(fig)
figure_paths.append(image_filter_path)

one_d = np.array([1, 4, 6, 4, 1], dtype=float)
one_d /= one_d.sum()
outer_kernel = np.outer(one_d, one_d)
patch = image[80:116, 90:126]
full_2d = sp_signal.convolve2d(patch, outer_kernel, mode="same", boundary="symm")
sep_2d = ndimage.convolve1d(ndimage.convolve1d(patch, one_d, axis=1, mode="reflect"), one_d, axis=0, mode="reflect")

image_checks = {
    "input_variance": float(np.var(image)),
    "box_blur_variance": float(np.var(box_blur)),
    "gaussian_blur_variance": float(np.var(gaussian_blur)),
    "unsharp_highpass_energy": float(np.mean((unsharp - gaussian_blur) ** 2)),
    "input_highpass_energy": float(np.mean((image - gaussian_blur) ** 2)),
    "separable_vs_full_2d_max_error": float(np.max(np.abs(full_2d - sep_2d))),
}
invariant_checks["image_filtering"] = image_checks
assert image_checks["gaussian_blur_variance"] < image_checks["input_variance"]
assert image_checks["box_blur_variance"] < image_checks["input_variance"]
assert image_checks["unsharp_highpass_energy"] > image_checks["input_highpass_energy"]
assert image_checks["separable_vs_full_2d_max_error"] < 1e-12
display_artifact(image_filter_path, width=900)
image_checks

## 6. Resampling: reconstruction and sampling collapse into one filter

Resizing an image is not just choosing which old pixels to keep. Conceptually, the input samples define a continuous reconstructed image, and the output grid samples that function. When shrinking, the output grid is coarser, so the filter must widen enough to low-pass the signal before the new samples are taken. This is why a reconstruction-sized tent filter is not enough for serious downsampling.

The resampler below uses the coordinate convention from earlier raster chapters: the whole input spans `[-0.5, width - 0.5]` in sample coordinates, and output sample locations are placed at the centers of equal-width cells in that source interval. Boundaries use edge extension and per-sample weight normalization to preserve constants near the edge.

In [ ]:
def zone_plate(n=288):
    y, x = np.mgrid[0:n, 0:n]
    X = (x - n / 2) / (n / 2)
    Y = (y - n / 2) / (n / 2)
    radial = 0.5 + 0.5 * np.cos(118.0 * (X**2 + Y**2))
    bars = 0.5 + 0.5 * np.sin(52.0 * (0.93 * X - 0.2 * Y))
    edge = (X - 0.18 * Y > 0.05).astype(float)
    return np.clip(0.50 * radial + 0.30 * bars + 0.20 * edge, 0.0, 1.0)

source = zone_plate(288)
target_shape = (96, 96)
nearest = resample_image(source, target_shape, lambda z: box_filter(z, 0.5), radius=0.5, filter_scale=1.0)
tent_unscaled = resample_image(source, target_shape, tent_filter, radius=1.0, filter_scale=1.0)
mitchell_scaled = resample_image(source, target_shape, mitchell_netravali_filter, radius=2.0, filter_scale=None)
gaussian_scaled = resample_image(source, target_shape, lambda z: gaussian_filter_1d(z, sigma=1.0), radius=3.0, filter_scale=None)

def spectrum_view(arr):
    centered = arr - arr.mean()
    spec = np.log1p(np.abs(fftshift(fft2(centered))))
    return normalize01(spec)

resampled_images = [nearest, tent_unscaled, mitchell_scaled, gaussian_scaled]
resampled_titles = ["nearest / pixel dropping", "tent at reconstruction scale", "scaled Mitchell-Netravali", "scaled Gaussian"]
spectra = [spectrum_view(arr) for arr in resampled_images]

fig, axes = plt.subplots(3, 4, figsize=(12.0, 9.2), constrained_layout=True)
source_crop = source[40:248, 40:248]
for col in range(4):
    axes[0, col].imshow(source_crop, cmap="gray", vmin=0, vmax=1)
    axes[0, col].set_title("source crop" if col == 0 else "")
    axes[0, col].set_xticks([])
    axes[0, col].set_yticks([])
for ax, arr, title in zip(axes[1], resampled_images, resampled_titles):
    ax.imshow(arr, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
for ax, arr, title in zip(axes[2], spectra, ["spectrum: " + t.split(" /")[0] for t in resampled_titles]):
    ax.imshow(arr, cmap="magma", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

resampling_path = save_matplotlib(fig, TOPIC, "resampling-antialiasing-comparison.png")
close(fig)
figure_paths.append(resampling_path)

constant = np.ones((33, 41), dtype=float) * 0.37
constant_resampled = resample_image(constant, (19, 23), mitchell_netravali_filter, radius=2.0, filter_scale=None)
resampling_checks = {
    "nearest_high_frequency_energy": high_frequency_energy(nearest),
    "tent_unscaled_high_frequency_energy": high_frequency_energy(tent_unscaled),
    "mitchell_scaled_high_frequency_energy": high_frequency_energy(mitchell_scaled),
    "gaussian_scaled_high_frequency_energy": high_frequency_energy(gaussian_scaled),
    "constant_resample_max_error": float(np.max(np.abs(constant_resampled - 0.37))),
    "downsample_factor": float(source.shape[0] / target_shape[0]),
}
invariant_checks["resampling"] = resampling_checks
assert resampling_checks["mitchell_scaled_high_frequency_energy"] < resampling_checks["nearest_high_frequency_energy"]
assert resampling_checks["gaussian_scaled_high_frequency_energy"] < resampling_checks["nearest_high_frequency_energy"]
assert resampling_checks["constant_resample_max_error"] < 1e-12
display_artifact(resampling_path, width=920)
resampling_checks

## 7. Fourier-domain scaffold: convolution becomes multiplication

The chapter's deepest simplification is the convolution theorem: a convolution in the signal domain becomes multiplication in the frequency domain. The notebook checks the discrete analogue with zero-padded FFTs. This is also why filter plots in frequency are useful: they show which parts of a signal survive the convolution.

The HTML artifact below is intentionally simple. Toggle the sample-rate cases and inspect how the alias copies move relative to the Nyquist band. The base spectrum is synthetic, but the geometry of the argument is the chapter's point: sampling creates translated spectra, and filtering tries to keep those translations out of the band we will reconstruct.

In [ ]:
rng = np.random.default_rng(10)
signal_a = rng.normal(size=19)
signal_b = np.array([0.0, 0.15, 0.7, 0.15, 0.0])
linear = sp_signal.convolve(signal_a, signal_b, mode="full")
fft_size = len(linear)
via_fft = irfft(rfft(signal_a, fft_size) * rfft(signal_b, fft_size), fft_size)
convolution_theorem_error = float(np.max(np.abs(linear - via_fft)))

sample_rates = [0.85, 1.2, 2.0]
labels = ["undersampled", "borderline", "higher sample rate"]
fig = go.Figure()
for group_index, (fs, label) in enumerate(zip(sample_rates, labels)):
    base, alias, total = alias_components(fs, filtered=False)
    _, _, filtered_total = alias_components(fs, filtered=True)
    traces = [
        go.Scatter(x=u, y=base, mode="lines", name=f"base, {label}", line={"color": "#2f6fbb", "width": 2}),
        go.Scatter(x=u, y=alias, mode="lines", name=f"alias copies, {label}", line={"color": "#c44e52", "width": 1.5}),
        go.Scatter(x=u, y=filtered_total, mode="lines", name=f"after low-pass, {label}", line={"color": "#2a9d8f", "width": 2, "dash": "dash"}),
    ]
    for trace in traces:
        trace.visible = group_index == 0
        fig.add_trace(trace)

buttons = []
for group_index, (fs, label) in enumerate(zip(sample_rates, labels)):
    visible = [False] * (3 * len(sample_rates))
    for j in range(3):
        visible[3 * group_index + j] = True
    buttons.append({
        "label": f"{label}: fs={fs}",
        "method": "update",
        "args": [
            {"visible": visible},
            {"title": f"Sampling spectrum copies: {label} (sample rate {fs})"},
        ],
    })

fig.add_vrect(x0=-0.5, x1=0.5, fillcolor="#dbeafe", opacity=0.28, line_width=0, annotation_text="unit Nyquist band", annotation_position="top left")
fig.update_layout(
    title="Sampling spectrum copies: undersampled (sample rate 0.85)",
    xaxis_title="frequency, cycles per unit",
    yaxis_title="relative magnitude",
    width=900,
    height=470,
    template="plotly_white",
    updatemenus=[{"type": "buttons", "buttons": buttons, "direction": "right", "x": 0.02, "y": 1.16}],
    legend={"orientation": "h", "y": -0.22},
    margin={"l": 55, "r": 25, "t": 90, "b": 100},
)
html_path = save_plotly_html(fig, TOPIC, "sampling-spectrum-alias-lab.html")
html_paths.append(html_path)

fourier_checks = {
    "linear_convolution_theorem_max_error": convolution_theorem_error,
    "html_artifact": book_path(html_path),
}
invariant_checks["fourier_convolution"] = fourier_checks
assert convolution_theorem_error < 1e-12
display_artifact(html_path, width="100%", height=520)
fourier_checks

## Applied lab

Use the resampling code as a design lab rather than as a black box.

1. Change the target shape from `(96, 96)` to `(144, 144)` and predict whether the scaled Mitchell filter should use a smaller or larger support. Then compare the spectra.
2. Replace the Mitchell-Netravali filter with Catmull-Rom in the downsampling example. Look for sharper edges and more ringing; then check whether the high-frequency energy increased.
3. Set `filter_scale=1.0` for every downsampling method. This intentionally treats shrinking like reconstruction only. The image should look sharper, but the Fourier view should show more energy near the edge of the spectrum.
4. Try a constant image and verify it remains constant. If it does not, the boundary mode or weight normalization is wrong.

A good lab note names both a visual change and a numeric change. In signal processing, a picture without an invariant is easy to overread, and an invariant without a picture is easy to misinterpret.

In [ ]:
invariant_path = save_json(invariant_checks, TOPIC, "signal-processing-invariants.json")
numeric_path = save_json(invariant_checks, TOPIC, "numeric-checks.json")
check_paths.extend([invariant_path, numeric_path])

display_artifact(invariant_path)
display_artifact(numeric_path)

summary_rows = [
    {"check": "aliased sample equality", "value": invariant_checks["sampling"]["alias_sample_max_error"], "expected": "< 1e-12"},
    {"check": "convolution commutativity", "value": invariant_checks["convolution"]["commutative_max_error"], "expected": "< 1e-12"},
    {"check": "convolution associativity", "value": invariant_checks["convolution"]["associative_max_error"], "expected": "< 1e-12"},
    {"check": "tent ripple span", "value": invariant_checks["filters"]["tent_ripple_span"], "expected": "< 1e-12"},
    {"check": "low-pass alias overlap ratio", "value": invariant_checks["frequency_response"]["lowpass_overlap_ratio"], "expected": "< 0.55"},
    {"check": "separable vs full 2D convolution", "value": invariant_checks["image_filtering"]["separable_vs_full_2d_max_error"], "expected": "< 1e-12"},
    {"check": "constant resampling preservation", "value": invariant_checks["resampling"]["constant_resample_max_error"], "expected": "< 1e-12"},
    {"check": "FFT convolution theorem", "value": invariant_checks["fourier_convolution"]["linear_convolution_theorem_max_error"], "expected": "< 1e-12"},
]
check_summary = pd.DataFrame(summary_rows)
check_summary

## Sanity checks

These final checks are deliberately redundant with the local assertions above. They make the notebook safe to run with `nbclient`: every declared artifact must exist, images must be nonblank, and the core chapter identities must still hold after re-execution.

In [ ]:
all_artifacts = [*figure_paths, *html_paths, *table_paths, *check_paths]
artifact_records = assert_artifacts(all_artifacts, min_bytes=1200)
image_records = [assert_nonblank_image(path) for path in figure_paths]

assert invariant_checks["sampling"]["alias_sample_max_error"] < 1e-12
assert invariant_checks["convolution"]["commutative_max_error"] < 1e-12
assert invariant_checks["convolution"]["associative_max_error"] < 1e-12
assert invariant_checks["frequency_response"]["lowpass_overlap_ratio"] < 0.55
assert invariant_checks["image_filtering"]["gaussian_blur_variance"] < invariant_checks["image_filtering"]["input_variance"]
assert invariant_checks["resampling"]["mitchell_scaled_high_frequency_energy"] < invariant_checks["resampling"]["nearest_high_frequency_energy"]
assert invariant_checks["fourier_convolution"]["linear_convolution_theorem_max_error"] < 1e-12

final_report = {
    "chapter": CHAPTER,
    "title": TITLE,
    "source_span": {"printed_pages": PRINTED_PAGES, "pdf_pages": PDF_PAGES},
    "figure_count": len(figure_paths),
    "html_count": len(html_paths),
    "table_count": len(table_paths),
    "check_count": len(check_paths),
    "artifact_count": len(all_artifacts),
    "nonblank_image_count": len(image_records),
    "artifact_records": artifact_records,
    "image_records": image_records,
    "core_checks": {
        "alias_sample_max_error": invariant_checks["sampling"]["alias_sample_max_error"],
        "convolution_associative_max_error": invariant_checks["convolution"]["associative_max_error"],
        "lowpass_overlap_ratio": invariant_checks["frequency_response"]["lowpass_overlap_ratio"],
        "mitchell_scaled_high_frequency_energy": invariant_checks["resampling"]["mitchell_scaled_high_frequency_energy"],
        "nearest_high_frequency_energy": invariant_checks["resampling"]["nearest_high_frequency_energy"],
        "fft_convolution_theorem_error": invariant_checks["fourier_convolution"]["linear_convolution_theorem_max_error"],
    },
}
final_path = save_json(final_report, TOPIC, "final-sanity.json")
display_artifact(final_path)
final_report

## Takeaways

- Sampling stores measurements, not the original function. If frequencies exceed the Nyquist limit, aliases are already mixed into the samples.
- Convolution is the common language for smoothing, reconstruction, image filtering, shadows, sharpening, and resampling.
- Normalized filters preserve average intensity; ripple-free reconstruction preserves constants at fractional positions.
- Filter choice is a tradeoff among cost, support, smoothness, ringing, interpolation, and frequency leakage.
- Downsampling must use a filter scaled to the output spacing. Pixel dropping is fast, but it keeps high-frequency structure that the new grid cannot represent.
- Fourier plots are diagnostic tools: low-pass filtering narrows spectra, sampling repeats spectra, and reconstruction filters try to keep alias copies out of the base band.